In [1]:
import numpy as np
import torch
import os
import sys
import h5py
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.insert(0, parent_dir)
from model import AttentionModel
from model_PCA_correlation import AttentionModel_PCA
from model_PCA_cond import ModelPCAcondJ
from dcascore import *
from utils import read_fasta_alignment, remove_duplicate_sequences, add_PCA_coords,quickread

# back to original path (in PLM)
sys.path.pop(0)  # Removes the parent_dir from sys.path
from model import AttentionModel

from plm_gen_methods import generate_plm_n_save, generate_coords_n_save, generate_multiple_targets_n_save,generate_plm_vect_n_save
from seq_utils import read_tensor_from_txt, set_seed, letters_to_nums, modify_seq ,sequences_from_fasta


**Generate sequences without PCA to verify the correlations (PLM and Ar attention)**

*Python Training (Alessandro and modified code)*

In [7]:
#Alessandro 
"""
    Load Q, K, V matrices from jdoms (after training)
"""
set_seed()
H = 64
d= 10
N = 174
n_epochs = 500
nb_PCA_comp=2
loss_type = 'without_J'
family = 'jdoms' #'jdoms_bacteria_train2'
cwd = parent_dir
Q_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_youss/Q_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
K_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_youss/K_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
V_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_youss/V_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
H,d,N=Q_1.shape
q=V_1.shape[1]
model=AttentionModel(H,d,N,q,Q=Q_1,V=V_1,K=K_1)
torch.sum(model.Q-Q_1)
device = Q_1.device
L = Q_1.shape[-1]
W=attention_heads_from_model(model,Q_1,K_1,V_1)
print(W.shape)

i_indices = torch.arange(L, device=device).unsqueeze(1)
j_indices = torch.arange(L, device=device).unsqueeze(0)
mask = (i_indices != j_indices).float().unsqueeze(0)  # shape (1, L, L)
W = W * mask
    
# Compute Jtens
Jtens = torch.einsum('hri,hab->abri', W, V_1)  # Shape: (q, q, L, L)
J_aless = torch.einsum('hri,hab->abri', W, V_1)
q = Jtens.shape[0]
N = Jtens.shape[2]
print(q)
print(N)
print(Jtens.shape)

torch.Size([64, 63, 63])
21
63
torch.Size([21, 21, 63, 63])


In [4]:
save_dir = "plm_gen_vectorized"
N_seqs = 30000

n_iter=1500
nb_seq=10000
save_name = f"alessandro_plm_randinit_n_iter{n_iter}_nb_seq{nb_seq}_b_1"
generate_plm_vect_n_save(save_dir,save_name,Jtens,n_iter,nb_seq,beta=1)

100%|██████████| 1500/1500 [08:14<00:00,  3.04it/s]

Generated sequences (letters): ['-YWSAFGITPEASKAEIAAAYRDLVARHHPDNA-DSSIAAESMARVKAAYQALMD--------', 'DHYKILDVKPYASLHDIKAAYMQRAKRLHPDAN-HNQAARDHFAASSPAYSVLSRAAKRHAYD', '-SYLILGLTRMASEEEIKRAYRELAKRYHPDSNPNDFRAEESFRRLLDAYQVLKRA-------', '----ILGLPNTASFAEVDKAYRELADKYHPDYNKEHGEAEEEFRKINEAYEVLT---------', 'DHYEVLGVAPTAGSADIKTCYRTLAGQHHPDSRGYKQASEEVLKDINEAYEVLKNKTTREAYD']
Generated sequences saved to plm_gen_vectorized


In [16]:
#Modified
H = 64
d= 10
N = 174
n_epochs = 500
nb_PCA_comp=2
loss_type = 'without_J'
family = 'jdoms' #'jdoms_bacteria_train2'
cwd = parent_dir
Q_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_mod_alessandro/Q_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
K_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_mod_alessandro/K_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
V_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_mod_alessandro/V_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
H,d,N=Q_1.shape
q=V_1.shape[1]
model=AttentionModel(H,d,N,q,Q=Q_1,V=V_1,K=K_1)
torch.sum(model.Q-Q_1)
device = Q_1.device
L = Q_1.shape[-1]
W=attention_heads_from_model(model,Q_1,K_1,V_1)
print(W.shape)

i_indices = torch.arange(L, device=device).unsqueeze(1)
j_indices = torch.arange(L, device=device).unsqueeze(0)
mask = (i_indices != j_indices).float().unsqueeze(0)  # shape (1, L, L)
W = W * mask
    
# Compute Jtens
Jtens = torch.einsum('hri,hab->abri', W, V_1)  # Shape: (q, q, L, L)
J_mod = torch.einsum('hri,hab->abri', W, V_1)
q = Jtens.shape[0]
N = Jtens.shape[2]
print(q)
print(N)
print(Jtens.shape)
print(Jtens.shape[-1])

torch.Size([64, 63, 63])
21
63
torch.Size([21, 21, 63, 63])
63


In [ ]:
save_dir = "plm_gen_vectorized"


n_iter=1500
nb_seq=10000
save_name = f"modified_plm_randinit_n_iter{n_iter}_nb_seq{nb_seq}_b_1"
generate_plm_vect_n_save(save_dir,save_name,Jtens,n_iter,nb_seq,beta=1)

100%|██████████| 1500/1500 [09:22<00:00,  2.66it/s]

Generated sequences (letters): ['DYHDILGVASDASTPEIKAAHKSLILKYHPDRMTEFTEAEILARMLNDARQVLLDPDSRAQY-', '-PYRDLDVSIDASSTEIKVSYRRAALEQHPDSPRKDTSGIERMQAVNIAKEILRDAIKRLKFD', 'EYYEVLEVDWYAGNEQIKKAYRKLASEYHPDKNQGNAAAVTRFKEIQEAYETLSDPNKREVYD', '--FGVLPIEPAADFQELHTAWRKLQQVFHPDVN-RSHF-AIQIQSINTSYDVLKKYDSRRT--', 'QWQGYDCSNNRR-HTLDILQDS-EIQQMSKVFYSTKFNYKKLKYPHMAKAHGFCSKVT-ERKE']
Generated sequences saved to plm_gen_vectorized


*Import Julia results*

In [ ]:
from scipy.special import softmax
set_seed()
H = 64
d= 10
N = 174
q=21
n_epochs = 500
nb_PCA_comp=2
L=63
loss_type = 'without_J'
family = 'jdoms' #'jdoms_bacteria_train2'
import h5py

f = h5py.File("matrices.jld2", "r")
Q_1 = f["Q"][:]
K_1 = f["K"][:]
V_1 = f["V"][:]
J = f["J"][:]
cwd = parent_dir
# Q_1 = np.loadtxt( cwd +"/results/Pagnani_julia_results/Q_tensor.txt")
# K_1 = np.loadtxt( cwd +"/results/Pagnani_julia_results/K_tensor.txt")
# V_1 = np.loadtxt( cwd +"/results/Pagnani_julia_results/V_tensor.txt")
print(Q_1.shape)
Q_1=torch.from_numpy(Q_1)
Q_1=Q_1.permute(2,1,0)
print(Q_1.shape)
K_1=torch.from_numpy(K_1)
K_1=K_1.permute(2,1,0)
print(K_1.shape)
V_1=torch.from_numpy(V_1)
V_1=V_1.permute(2,1,0)
print(V_1.shape)
J=torch.from_numpy(J)
print("jjj",J.shape)

J=J.permute(1,0,3,2)

model=AttentionModel(H,d,N,q,Q=Q_1,V=V_1,K=K_1)
torch.sum(model.Q-Q_1)
device = Q_1.device
L = Q_1.shape[-1]
W=attention_heads_from_model(model,Q_1,K_1,V_1)

print(W.shape)
J
i_indices = torch.arange(L, device=device).unsqueeze(1)
j_indices = torch.arange(L, device=device).unsqueeze(0)
mask = (i_indices != j_indices).float().unsqueeze(0)  # shape (1, L, L)
W = W * mask

W = W.double()

V_1 = V_1.double()
#Compute Jtens
sf=np.zeros((H,L,L))
perso_mask=np.ones((L,L))-np.eye(L)
for h in range(H):
    sf[h,:,:]=torch.softmax(torch.from_numpy(np.einsum("di,dj->ij",Q_1[h,:,:],K_1[h,:,:])),axis=1)

J_tens_by_hand=np.einsum("hab,hij,ij->abij",V_1,sf,perso_mask)
Jtens = torch.einsum('hri,hab->abri', W, V_1)  # Shape: (q, q, L, L)
# Shape: (q, q, L, L)
J_julia = torch.einsum('hri,hab->abri', W, V_1)
print(Jtens.shape)
diff=(J-Jtens.numpy())**2
print("difference between tensors",diff.mean().item())



(63, 10, 64)
torch.Size([64, 10, 63])
torch.Size([64, 10, 63])
torch.Size([64, 21, 21])
jjj torch.Size([21, 21, 63, 63])
torch.Size([64, 63, 63])
WW torch.Size([64, 63, 63])
torch.Size([21, 21, 63, 63])
difference between tensors 3.6464975405611796e-18


C:\Users\youss\AppData\Local\Temp\ipykernel_36732\487856659.py:61: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments.
  sf[h,:,:]=torch.softmax(torch.from_numpy(np.einsum("di,dj->ij",Q_1[h,:,:],K_1[h,:,:])),axis=1)
C:\Users\youss\AppData\Local\Temp\ipykernel_36732\487856659.py:68: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  diff=(J-Jtens.numpy())**2


0.012572310078934543
0.012572310078934506


In [10]:
save_dir = "plm_gen_vectorized"
N_seqs = 30000

n_iter=1500
nb_seq=10000
save_name = f"Julia_imported_QKV_plm_randinit_n_iter{n_iter}_nb_seq{nb_seq}_b_1"
generate_plm_vect_n_save(save_dir,save_name,Jtens,n_iter,nb_seq,beta=1)

100%|██████████| 1500/1500 [08:43<00:00,  2.86it/s]

Generated sequences (letters): ['DYYAILGVARDAGSKEIRDALKRLARKLHPDVNSGETEAEKLFRLVTAAWEVLKDPARRAEY-', '--FRILDVSEEDDRVSCKVSWNRILLILHPDRG----GSEQLVQAYNQAFKEL----------', 'DYYEVLGVASTAIPEHIRKAYRKLAREVHPDKNQGNAAAVVRFKILQEAYEVLSDPKKRATYD', '--YKMLNFGPAADIQEIKSAYRKLIQVFHPDTG---KG-ASKIQRINQSYDVLK---------', '-----FDLPLRL-EQEIHNRFRTLMMQYHPDKVRRDKLVLEKCTQINAAYHILKQKRS-A---']
Generated sequences saved to plm_gen_vectorized


We compare the probabilities generated between the different J tensors

In [10]:
from plm_model import BatchSequencePLM
from tqdm import tqdm
generated_dir = "plm_gen_vectorized"
n_iter=1500
nb_seq=10000
cwd = os.getcwd()
full_gen_path = os.path.join(cwd, generated_dir)
file= f"Julia_imported_QKV_plm_randinit_n_iter{n_iter}_nb_seq{nb_seq}_b_1.npy"
file_path = os.path.join(full_gen_path, file)
gen_sequences = np.load(file_path)
N,L=gen_sequences.shape
seqs=BatchSequencePLM(J_julia,N)
seqs.assign_seqs(gen_sequences)
seqs_alter=BatchSequencePLM(J_aless,N)
seqs_alter.assign_seqs(gen_sequences)
diff_Julia_mod=0
diff_julia_aless=0
diff_mod_aless=0
diff_0_normalement=0
for i in tqdm(range(L)):
    diff_julia_aless+=seqs.compare_J(J_aless,i)
    diff_Julia_mod+=seqs.compare_J(J_mod,i)
    diff_mod_aless+=seqs_alter.compare_J(J_mod,i)
    diff_0_normalement+=seqs.compare_J(J_julia,i)
print("prob difference J_julia and J_alessandro",diff_julia_aless/L)
print("prob difference J_julia and J_mod",diff_Julia_mod/L)
print("prob difference J_aless and J_mod",diff_mod_aless/L)
print("prob difference J_julia and J_Julia",diff_0_normalement)


  0%|          | 0/63 [00:00<?, ?it/s]

100%|██████████| 63/63 [01:05<00:00,  1.05s/it]

prob difference J_julia and J_alessandro 0.012555067277345323
prob difference J_julia and J_mod 0.012285435141707875
prob difference J_aless and J_mod 0.0064950380144695044
prob difference J_julia and J_Julia 0.0


**Comparing autoreg method python vs Julia**

tensors from Julia training 

In [2]:
from model import ArAttentionModel
def load_jld2(filename, key):
    with h5py.File(filename, "r") as f:
        data = f[key][:]  # NumPy array
    # Permute axes so indexing matches Julia
    return np.transpose(data, axes=range(data.ndim)[::-1])
H=64
d=10
N=63
q=21
# Example usage
Q_1 = load_jld2("arQ_tensor.jld2", "arQ")
K_1 = load_jld2("arK_tensor.jld2", "arK")
V_1 = load_jld2("arV_tensor.jld2", "arV")
J_julia = load_jld2("arJ_tensor.jld2", "arJ")
Q_1=torch.from_numpy(Q_1)
K_1=torch.from_numpy(K_1)
V_1=torch.from_numpy(V_1)
print(Q_1.shape)
print(J_julia.shape)
model=ArAttentionModel(H,d,N,q,Q=Q_1,V=V_1,K=K_1)
torch.sum(model.Q-Q_1)
device = Q_1.device
L = Q_1.shape[-1]
W=attention_heads_from_model(model,Q_1,K_1,V_1)
print(W.shape)

i_indices = torch.arange(L, device=device).unsqueeze(1)
j_indices = torch.arange(L, device=device).unsqueeze(0)
mask = (i_indices > j_indices).float().unsqueeze(0)  # shape (1, L, L)
W = W * mask
W = W.double()
V_1 = V_1.double()
# Compute Jtens
Jtens = torch.einsum('hri,hab->abri', W, V_1)  # Shape: (q, q, L, L)
J_julia_good_dim=np.transpose(J_julia,(2,3,0,1))
diff=(J_julia_good_dim-Jtens.numpy())**2
print("diff two tensors", diff.mean())
q = Jtens.shape[0]
N = Jtens.shape[2]
print(q)
print(N)
print(Jtens.shape)
print(Jtens[4,2,0,0])

torch.Size([64, 10, 63])
(63, 63, 21, 21)
torch.Size([64, 63, 63])
diff two tensors 3.6254347018872035e-18
21
63
torch.Size([21, 21, 63, 63])
tensor(0., dtype=torch.float64)


In [3]:
#calculate the probabilities of the first position
from gen_autoreg import SequenceAR
from Frquencies_plot_func import compute_position_frequencies
from seq_utils import letters_to_nums, sequences_from_fasta, one_hot_seq_batch
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.insert(0, parent_dir)
from model import AttentionModel
from model_PCA_correlation import AttentionModel_PCA
from dcascore import *
from utils import read_fasta_alignment, remove_duplicate_sequences, add_PCA_coords,quickread
family = 'jdoms_bacteria_train2'
file_test_data=r"C:\Users\youss\OneDrive\Bureau\master epfl\MA2\TP4 De los Rios\git_test\PLM-gen-DCA\Attention-DCA-main\CODE\DataAttentionDCA\jdoms\jdoms_bacteria_train2.fasta"
_,W=quickread(file_test_data)
W=W/W.sum()
train_sequences = sequences_from_fasta(file_test_data)
train_sequences = [seq.replace('.', '') for seq in train_sequences]  # remove all dots
    # remove non capital letters - '-' considered as capital letter
train_sequences = [''.join([c for c in seq if c.isupper() or c == '-']) for seq in train_sequences]
train_sequences_num = [letters_to_nums(seq) for seq in train_sequences]
train_sequences_num=np.array(train_sequences_num)

#generate autoreg
save_dir = "autoreg_gen"
save_name = "jdoms_J_julia"
p0=compute_position_frequencies(train_sequences_num,W)
nb_seq=10000
seq=SequenceAR(Jtens,nb_seq,p0[0,:],beta=1)
seq.gen_seq()
seq.save_sequence(save_dir,save_name)
gen_seq=seq.load_sequence(save_dir,save_name)

Total sequences read: 14502
Sequences after filtering: 14502
Sampling 100000 pairs out of 105146751 total pairs.
Mean fraction of identical positions (sampled): 0.37202728570888266
Computed theta: 0.3268577458459699


100%|██████████| 14502/14502 [00:09<00:00, 1543.40it/s]


3265.70225762244


 11%|█▏        | 7/62 [00:00<00:00, 69.32it/s]

(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 2)
torch.Size([10000, 2])
(10000, 2)
torch.Size([10000, 2])
(10000, 2)
torch.Size([10000, 2])
(10000, 2)
torch.Size([10000, 2])
(10000, 2)
torch.Size([10000, 2])
(10000, 2)
torch.Size([10000, 2])
(10000, 2)
torch.Size([10000, 2])
(10000, 2)
torch.Size([10000, 2])
(10000, 2)
tor

...

(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])


 98%|█████████▊| 61/62 [00:04<00:00,  7.56it/s]

(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 62)
torch.Size([10000, 62])
(10000, 62)
torch.Size([10000, 62])
(10000, 62)
torch.Size([10000, 62])
(10000, 62)
torch.Size([10000, 62])
(10000, 62)
torch.Size([10000, 62])
(10000, 62)
torch.Size([10000, 62])
(10000, 62)
torch.Size([1000

100%|██████████| 62/62 [00:04<00:00, 14.13it/s]


(10000, 62)
torch.Size([10000, 62])
(10000, 62)
torch.Size([10000, 62])
(10000, 62)
torch.Size([10000, 62])
(10000, 62)
torch.Size([10000, 62])
(10000, 62)
torch.Size([10000, 62])
(10000, 62)
torch.Size([10000, 62])
(10000, 62)
torch.Size([10000, 62])
(10000, 62)
torch.Size([10000, 62])


Tensors from Python training 

In [4]:
#Modified
H = 64
d= 10
N = 174
n_epochs = 500
nb_PCA_comp=2
loss_type = 'without_J'
family = 'jdoms' #'jdoms_bacteria_train2'
cwd = parent_dir
Q_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_AR_youss/Q_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
K_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_AR_youss/K_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
V_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_AR_youss/V_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
H,d,N=Q_1.shape
q=V_1.shape[1]
model=AttentionModel(H,d,N,q,Q=Q_1,V=V_1,K=K_1)
torch.sum(model.Q-Q_1)
device = Q_1.device
L = Q_1.shape[-1]
W=attention_heads_from_model(model,Q_1,K_1,V_1)
print(W.shape)

i_indices = torch.arange(L, device=device).unsqueeze(1)
j_indices = torch.arange(L, device=device).unsqueeze(0)
mask = (i_indices > j_indices).float().unsqueeze(0)  # shape (1, L, L)
W = W * mask
    
# Compute Jtens
Jtens = torch.einsum('hri,hab->abri', W, V_1)  # Shape: (q, q, L, L)
J_ar_py = torch.einsum('hri,hab->abri', W, V_1)
q = Jtens.shape[0]
N = Jtens.shape[2]
print(q)
print(N)
print(Jtens.shape)
print(Jtens.shape[-1])

torch.Size([64, 63, 63])
21
63
torch.Size([21, 21, 63, 63])
63


In [5]:
#calculate the probabilities of the first position
from gen_autoreg import SequenceAR
from Frquencies_plot_func import compute_position_frequencies
from seq_utils import letters_to_nums, sequences_from_fasta, one_hot_seq_batch
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.insert(0, parent_dir)
from model import AttentionModel
from model_PCA_correlation import AttentionModel_PCA
from dcascore import *
from utils import read_fasta_alignment, remove_duplicate_sequences, add_PCA_coords,quickread
family = 'jdoms_bacteria_train2'
file_test_data=r"C:\Users\youss\OneDrive\Bureau\master epfl\MA2\TP4 De los Rios\git_test\PLM-gen-DCA\Attention-DCA-main\CODE\DataAttentionDCA\jdoms\jdoms_bacteria_train2.fasta"
_,W=quickread(file_test_data)
W=W/W.sum()
train_sequences = sequences_from_fasta(file_test_data)
train_sequences = [seq.replace('.', '') for seq in train_sequences]  # remove all dots
    # remove non capital letters - '-' considered as capital letter
train_sequences = [''.join([c for c in seq if c.isupper() or c == '-']) for seq in train_sequences]
train_sequences_num = [letters_to_nums(seq) for seq in train_sequences]
train_sequences_num=np.array(train_sequences_num)

#generate autoreg
save_dir = "autoreg_gen"
save_name = "jdoms_J_python"
p0=compute_position_frequencies(train_sequences_num,W)
nb_seq=10000
seq=SequenceAR(Jtens,nb_seq,p0[0,:],beta=1)
seq.gen_seq()
seq.save_sequence(save_dir,save_name)
gen_seq=seq.load_sequence(save_dir,save_name)

Total sequences read: 14502
Sequences after filtering: 14502
Sampling 100000 pairs out of 105146751 total pairs.
Mean fraction of identical positions (sampled): 0.37202728570888266
Computed theta: 0.3268577458459699


100%|██████████| 14502/14502 [00:09<00:00, 1482.79it/s]


3265.70225762244


 13%|█▎        | 8/62 [00:00<00:00, 73.73it/s]

(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 1)
torch.Size([10000, 1])
(10000, 2)
torch.Size([10000, 2])
(10000, 2)
torch.Size([10000, 2])
(10000, 2)
torch.Size([10000, 2])
(10000, 2)
torch.Size([10000, 2])
(10000, 2)
torch.Size([10000, 2])
(10000, 2)
torch.Size([10000, 2])
(10000, 2)
torch.Size([10000, 2])
(10000, 2)
torch.Size([10000, 2])
(10000, 2)
tor

...

(10000, 56)
torch.Size([10000, 56])
(10000, 56)
torch.Size([10000, 56])
(10000, 56)
torch.Size([10000, 56])
(10000, 56)
torch.Size([10000, 56])
(10000, 56)
torch.Size([10000, 56])
(10000, 56)
torch.Size([10000, 56])
(10000, 56)
torch.Size([10000, 56])
(10000, 56)
torch.Size([10000, 56])
(10000, 57)
torch.Size([10000, 57])
(10000, 57)
torch.Size([10000, 57])
(10000, 57)
torch.Size([10000, 57])
(10000, 57)
torch.Size([10000, 57])
(10000, 57)
torch.Size([10000, 57])
(10000, 57)
torch.Size([10000, 57])
(10000, 57)
torch.Size([10000, 57])
(10000, 57)
torch.Size([10000, 57])
(10000, 57)
torch.Size([10000, 57])
(10000, 57)
torch.Size([10000, 57])
(10000, 57)
torch.Size([10000, 57])
(10000, 57)
torch.Size([10000, 57])
(10000, 57)
torch.Size([10000, 57])
(10000, 57)
torch.Size([10000, 57])
(10000, 57)
torch.Size([10000, 57])
(10000, 57)
torch.Size([10000, 57])
(10000, 57)
torch.Size([10000, 57])
(10000, 57)
torch.Size([10000, 57])
(10000, 57)
torch.Size([10000, 57])
(10000, 57)
torch.Size([1000

 94%|█████████▎| 58/62 [00:03<00:00, 11.14it/s]

torch.Size([10000, 58])
(10000, 58)
torch.Size([10000, 58])
(10000, 58)
torch.Size([10000, 58])
(10000, 58)
torch.Size([10000, 58])
(10000, 58)
torch.Size([10000, 58])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 59)
torch.Size([10000, 59])
(10000, 60)
torch.Size([10000, 60])
(10000, 60)
torch.Size([10000, 60])
(100

100%|██████████| 62/62 [00:03<00:00, 18.02it/s]


(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 61)
torch.Size([10000, 61])
(10000, 62)
torch.Size([10000, 62])
(10000, 62)
torch.Size([10000, 62])
(10000, 62)
torch.Size([10000, 62])
(10000, 62)
torch.Size([10000, 62])
(10000, 62)
torch.Size([10000, 62])
(10000, 62)
torch.Size([10000, 62])
(10000, 62)
torch.Size([1000

***PCA coord generation*** only 2 PCA components for now

Brute force: J.shape [35, 35, 65, 65]

In [15]:
"""
    Load Q, K, V matrices from jdoms (after training)
"""
set_seed()
H = 64
d= 10
N = 174
n_epochs = 500
nb_PCA_comp=2
loss_type = 'without_J'
family = 'jdoms' #'jdoms_bacteria_train2'
cwd = parent_dir
Q_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA_brute_force_35_bins/Q_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
K_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA_brute_force_35_bins/K_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
V_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_PCA_brute_force_35_bins/V_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
H,d,N=Q_1.shape
q=V_1.shape[1]
model=AttentionModel(H,d,N,q,Q=Q_1,V=V_1,K=K_1)
torch.sum(model.Q-Q_1)
device = Q_1.device
L = Q_1.shape[-1]
W=attention_heads_from_model(model,Q_1,K_1,V_1)
print(W.shape)

i_indices = torch.arange(L, device=device).unsqueeze(1)
j_indices = torch.arange(L, device=device).unsqueeze(0)
mask = (i_indices != j_indices).float().unsqueeze(0)  # shape (1, L, L)
W = W * mask
    
# Compute Jtens
Jtens = torch.einsum('hri,hab->abri', W, V_1)  # Shape: (q, q, L, L)
q = Jtens.shape[0]
N = Jtens.shape[2]
print(q)
print(N)
print(Jtens.shape)


torch.Size([64, 65, 65])
35
65
torch.Size([35, 35, 65, 65])


In [ ]:
print(type(Jtens[0,np.full(100,1),0,0]))

In [ ]:
save_dir = "generated_vect_sequences_brute_force"
N_seqs = 30000

n_iter=1500
nb_seq=10000
save_name = f"gen_vect_plm_randinit_n_iter{n_iter}_nb_seq{nb_seq}_b_PCA_15_b_1_2_PCA24_20"
generate_plm_vect_n_save(save_dir,save_name,Jtens,n_iter,nb_seq,beta=1.2,nb_PCA_comp=nb_PCA_comp,beta_PCA=15,PCA_comp_list=np.array([24,20]))

2 model (two tensors one for self sequence intereaction and one for interaction with with PCA components) with pretrained J_interaction

In [2]:
H = 64
d= 10
N = 174
n_epochs = 500
nb_PCA_comp=2
loss_type = 'without_J'
family = 'jdoms' #'jdoms_bacteria_train2'
cwd = parent_dir
Q_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_mod_alessandro/Q_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
K_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_mod_alessandro/K_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
V_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_mod_alessandro/V_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
H,d,N=Q_1.shape
q=V_1.shape[1]
model=AttentionModel(H,d,N,q,Q=Q_1,V=V_1,K=K_1)
torch.sum(model.Q-Q_1)
device = Q_1.device
L = Q_1.shape[-1]
W=attention_heads_from_model(model,Q_1,K_1,V_1)
print(W.shape)

i_indices = torch.arange(L, device=device).unsqueeze(1)
j_indices = torch.arange(L, device=device).unsqueeze(0)
mask = (i_indices != j_indices).float().unsqueeze(0)  # shape (1, L, L)
W = W * mask
    
# Compute Jtens
Jtens = torch.einsum('hri,hab->abri', W, V_1)  # Shape: (q, q, L, L)
q = Jtens.shape[0]
N = Jtens.shape[2]
print(q)
print(N)
print(Jtens.shape)
print(Jtens.shape[-1])

torch.Size([64, 63, 63])
21
63
torch.Size([21, 21, 63, 63])
63


In [3]:
nb_PCA_comp=50 # to be changed 
Q_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_2model_pretrained_n_pca_{nb_PCA_comp}/Q_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs,nb_PCA_comp=nb_PCA_comp))
K_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_2model_pretrained_n_pca_{nb_PCA_comp}/K_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs,nb_PCA_comp=nb_PCA_comp))
V_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_2model_pretrained_n_pca_{nb_PCA_comp}/V_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs,nb_PCA_comp=nb_PCA_comp))
H,d,N1=Q_1.shape
_,_,N2=K_1.shape
_,q1,q2=V_1.shape
model=AttentionModel_PCA(H,d,N1,N2,q1,q2,Q=Q_1,V=V_1,K=K_1)
torch.sum(model.Q-Q_1)
device = Q_1.device
L = Q_1.shape[-1]
W=attention_heads_from_model(model,Q_1,K_1,V_1)
print(W.shape)
print(K_1.shape)
print(V_1.shape)
# Compute Jtens
Jtens_PCA = torch.einsum('hri,hab->abri', W, V_1)  # Shape: (q, q, L, L)
q = Jtens.shape[0]
N = Jtens.shape[2]
J_PCA2=Jtens_PCA**2
J_2=Jtens**2
print(J_PCA2.max().item())
print(J_2.max().item())
print(q)
print(N)
print(Jtens_PCA.shape)

torch.Size([64, 63, 50])
torch.Size([64, 10, 50])
torch.Size([64, 21, 17])
0.00020015222253277898
10.883213996887207
21
63
torch.Size([21, 17, 63, 50])


These next two cells are for finding some clusters in the PCA space of the train data so we can give them as targets to the generation part later 

In [2]:
# ----- Load train sequences -----
family = 'PF00014'
file_test_data=r"C:\Users\youss\OneDrive\Bureau\master epfl\MA2\TP4 De los Rios\PCA_gen_ai\PCA_gen_AI\CODE\DataAttentionDCA\PF00014\PF00014.fasta"
family = 'jdoms_bacteria_train2'
file_test_data=r"C:\Users\youss\OneDrive\Bureau\master epfl\MA2\TP4 De los Rios\git_test\PLM-gen-DCA\Attention-DCA-main\CODE\DataAttentionDCA\jdoms\jdoms_bacteria_train2.fasta"
_,W=quickread(file_test_data)
W=W/W.sum()
print(W)
train_sequences = sequences_from_fasta(file_test_data)
train_sequences = [seq.replace('.', '') for seq in train_sequences]  # remove all dots
    # remove non capital letters - '-' considered as capital letter
train_sequences = [''.join([c for c in seq if c.isupper() or c == '-']) for seq in train_sequences]
train_sequences_num = [letters_to_nums(seq) for seq in train_sequences]
train_sequences_num=np.array(train_sequences_num)

Total sequences read: 14502
Sequences after filtering: 14502
Sampling 100000 pairs out of 105146751 total pairs.
Mean fraction of identical positions (sampled): 0.37145857142857136
Computed theta: 0.32735817491664143


100%|██████████| 14502/14502 [00:09<00:00, 1562.39it/s]


3265.70225762244
[4.93891696e-06 7.50521695e-07 2.18723465e-05 ... 1.02070950e-04
 1.02070950e-04 2.75867434e-06]


In [3]:
from utils import read_fasta_alignment, remove_duplicate_sequences, add_PCA_coords,quickread,pca_kmeans_analysis, plot_clusters_2d,get_PCA_grid_coords, get_proj_pca_coords, wasserstein_pca_distance
n_bins=35
results, pca, scaler=pca_kmeans_analysis(train_sequences_num,W,n_components=50,n_clusters=3,return_pca=True)
#plot_clusters_2d(results)
ov= get_PCA_grid_coords(results["centers"],n_bins,coords_min=results["min_pca_axe0"],coords_max=results["max_pca_axe0"])
print(results["centers"])
print(results["sizes"])
print(results["max_distances"])
print(ov)

[[ 6.55178993e+00  1.36233386e+01  1.05583858e+01  3.25951517e+00
   5.92315224e+00 -1.18160677e+00  1.24272145e+00 -1.72229747e-01
   9.94149161e-01  9.01120378e-03  4.72727721e-01  7.35518984e-02
  -4.70312460e-01  4.47802424e-01 -1.17093678e-02 -9.20412566e-01
   3.80232042e-01  2.41497275e-01 -4.54964774e-01 -3.05730912e-01
   1.65091191e-01  3.06141965e-01  2.27470364e-01  7.83886630e-01
  -3.69428357e-01 -9.70141591e-01  6.79101646e-01 -2.49250448e-01
  -4.14737991e-01  5.42170365e-01  9.42232109e-01  5.88548374e-01
   1.71057084e-01  4.03045067e-01 -4.88420639e-01 -3.49560533e-01
  -4.65519573e-01  3.44607453e-02 -6.06902554e-01 -2.40718123e-01
  -4.39422085e-02  4.34539432e-01 -1.22298877e-01  6.16130875e-01
  -2.05288247e-03  2.51481480e-02  1.16821498e-01 -1.88641233e-01
  -1.64132040e-01 -2.05948246e-02]
 [ 5.59371729e+00 -8.15434959e-01 -3.32516830e-01 -5.14801765e-01
  -4.45457268e-01 -6.75164581e-02 -1.47674253e-01 -2.68401289e-03
  -8.96760060e-02 -2.81209435e-02  1.1145

In [ ]:

save_dir = f"generated_vect_sequences_2model_pretrained_n_PCA{nb_PCA_comp}_n_bins{n_bins}"
n_iter=1200
nb_seq=5000
target=np.array(ov[0])
betas=[1,5,10,50]#,0.1,0.5,1,5,10]
for beta in betas:
    save_name = f"plm_randinit_n_iter{n_iter}_nb_seq{nb_seq}_b_PCA_{beta}_b_1_nb_PCA{nb_PCA_comp}"
    generate_plm_vect_n_save(save_dir,save_name,Jtens,n_iter,nb_seq,beta=1,nb_PCA_comp=nb_PCA_comp,beta_PCA=beta,PCA_comp_list=target,J_PCA=Jtens_PCA)

100%|██████████| 1200/1200 [05:21<00:00,  3.73it/s]


Generated sequences (letters): ['DPYHELGINRDATVEEQRTVNKNQVRALHPDQARNADPKTAKLASFAKAAGVNSDRAKKKMPD', 'DPYQLLGVSPMASLDEAKRRYLDLMKRHHPDSQQGNPIAEKRLVEIPAAY-------------', '-FYDILELGSGATNVEAKTAYRRLLAEYHPDKNSDHVWASEKMRQINAAYEELKDSDKRHLYD', 'NHYRVLGLEITADQDPIKEQYRKLLALYHPDRHPEHQLATRKMAHVNAAYSVLKTPDKRARYD', 'DYYAILEVTRDASAEDVKAAYLRLAQDAHPDKH-KRPEAESRFREIQRAFDVLRDPERRSRYD']
Generated sequences saved to generated_vect_sequences_2model_pretrained_n_PCA50_n_bins17PCA_comp


100%|██████████| 1200/1200 [05:55<00:00,  3.38it/s]


Generated sequences (letters): ['DFYEVLGIDSDADEKEVKKAFRRLAKKHHPDLHGDDVGFEELFKEVVEASEVLSDVEKRRLYD', 'KEWFA-YDHEWDPGRRRHLACFSRYCCIGFLRVSSIEKRCWGNSNDDDEHQHQVGQVRYLDRR', 'NYYEILQVSKDAEMDEIKKAYRKLVQRYHPDKD--AIDIDERFKQVSEAYAVLSDPNKKAIYD', 'DPYRVLKVACGVTESAIRWRSKNLARKWLPDVRLQTAVTEDAFKKVQNAYRFLESKDNPKKND', '----KFGLQVMADKLAVARRFRELSRAAHPENW---MFDKQDMQYVEKSLQLLKDPLKRA---']
Generated sequences saved to generated_vect_sequences_2model_pretrained_n_PCA50_n_bins17PCA_comp


100%|██████████| 1200/1200 [05:48<00:00,  3.44it/s]


Generated sequences (letters): ['DYYDVMGLGENIDPAVLKKGKRALAHKFHPDAHTDGGQVEIVFGEISRAYQTLGDPWLRAIYD', '-YCDVLPIGPDAGEYVVKGAFVRSAKDNHPDTIR-AD-APDQFRRIQQAYRVLNDPQQRPAY-', 'DFYEVLGVSPDADQDAIKKAYRKRAKMFHPDHC--SGESA-VFQALKHAYDVLSDDDSRLPYD', '---SVLGLEEGSSPEQIKQSYKELAFMYHPDVN-NSPMAEERLKQIIAARDSLI---------', '-PYQILGLSPGASQREIKRATRRLVKRHHPDVNKGDVRAHVSMVRVQGAYEVLIKQ-------']
Generated sequences saved to generated_vect_sequences_2model_pretrained_n_PCA50_n_bins17PCA_comp


100%|██████████| 1200/1200 [05:49<00:00,  3.44it/s]


Generated sequences (letters): ['NYFVLLGLPIGNSSSDLDVQQRSLVKQFHPDFQAGDRQSSNFDVQINSAVQVLKGSSSDQNYN', 'NPYNIIGVPRQLSQRDIRQGFSKQMRLSGGDINMGGQGLQRVAGQVNVDMDQLSSRDSRRQSD', 'DPYRLLGIGRGGSCSDISAASQKQLGDPQADIDPSLFGAVQKAVVAQVSQGVLGDPDKRAQQD', 'LWGDLFGLQSDNRFDNLVSSGPRPVLHGHPDSLR--GLSVQGNAQGNDLWVQLGKDPQMGGG-', 'NPRVVMMWSQMIGGSNLRGLSSDLARADGLDRNGGDLDMIQIMVIWVLSLQVLGGQQDLGNVV']
Generated sequences saved to generated_vect_sequences_2model_pretrained_n_PCA50_n_bins17PCA_comp


100%|██████████| 1200/1200 [05:50<00:00,  3.43it/s]

Generated sequences (letters): ['GGGGVGGGGGGGGGGGLGGGGGGGVGGGGGDGGGGGGGSGGDGVGGGGGGGGDGGGGGGGGVG', 'GGGGVGGGGGGGGGGGGGGGGGGGVGGGGGDGGGGGGGSGGDGVGGGGGGGVDGGGGGGGGGG', 'GGGGVGGGGGGGGGGGGGGGGGGGVGGGGGDGGGGGGGSGGDMVGGGGGGGVGGGGGGGGGGG', 'GGGGGGGGGGGGGGGGGGGGGGGGDGGGGGDGGGGGGGSGGVGVGGGGGGGVDGGGGGGGGGG', 'GGGGVGGGGGGDGGGGGGGGGGGGDGGGGGDGGGGGGGSGGDGVGGGGVGGDGGGGGGGGGGG']
Generated sequences saved to generated_vect_sequences_2model_pretrained_n_PCA50_n_bins17PCA_comp


J (q,q+nbins,L,L+m)

In [ ]:
nb_PCA_comp=50
H = 64
d= 10
N = 174
n_epochs = 250
loss_type = 'without_J'
family = 'jdoms' #'jdoms_bacteria_train2'
cwd = parent_dir
Q_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_Jcond_PCA50_Nbins35/Q_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
K_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_Jcond_PCA50_Nbins35/K_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
V_1 = read_tensor_from_txt( cwd +"/results/{H}_{d}_{family}_{losstype}_{n_epochs}_Jcond_PCA50_Nbins35/V_tensor.txt".format(H=H, d=d, family=family, losstype=loss_type, n_epochs=n_epochs))
H,d,N=Q_1.shape
q=V_1.shape[1]
model=ModelPCAcondJ(H,d,N,q,Q=Q_1,V=V_1,K=K_1)
torch.sum(model.Q-Q_1)
device = Q_1.device
L = Q_1.shape[-1]
W=attention_heads_from_model(model,Q_1,K_1,V_1)
print(W.shape)

i_indices = torch.arange(L, device=device).unsqueeze(1)
j_indices = torch.arange(L, device=device).unsqueeze(0)
mask = (i_indices != j_indices).float().unsqueeze(0)  # shape (1, L, L)
W[:,:,:63] = W[:,:,:63] * mask
    
# Compute Jtens
Jtens = torch.einsum('hri,hab->abri', W, V_1)  # Shape: (q, q, L, L)
q = Jtens.shape[0]
N = Jtens.shape[2]
print(q)
print(N)
print(Jtens.shape)
print(Jtens.shape[-1])

torch.Size([64, 63, 113])
21
63
torch.Size([21, 56, 63, 113])
113


In [7]:
nb_PCA_comp=50
save_dir = f"generated_vect_sequences_Jcond_n_PCA{nb_PCA_comp}_n_bins{n_bins}"
n_iter=1200
nb_seq=5000
target=np.array(ov[0])
betas=[2]#[1,5,10,50]
for beta in betas:
    save_name = f"plm_randinit_n_iter{n_iter}_nb_seq{nb_seq}_b_PCA_{beta}_b_1_nb_PCA{nb_PCA_comp}"
    generate_plm_vect_n_save(save_dir,save_name,Jtens,n_iter,nb_seq,beta=1,nb_PCA_comp=nb_PCA_comp,beta_PCA=beta,PCA_comp_list=target)

50
50


100%|██████████| 1200/1200 [50:54<00:00,  2.55s/it]    

Generated sequences (letters): ['DPYTVLGIAKDATAADIRKAYRRLAKEHHPDRL--EPDSITRFQELSKAYEQLEEPKKRNEYD', 'KFYDILGIARTASRDDIHSAYRRLAKLYHPDTNQEWRLSHEKFLEITDAYAVLSDQEKRQEYD', 'DYYEVLGVSRHATAAEIRRQYRLLALLSHPDRRENETASEDRFREITAAYETLRDEDSRARYD', 'NYFEVLGLAKKADRADIAKAYRTIAKLTHPDTL-MVTKSETKFAEIAHAYEWLTDEMQRYRY-', 'DPFRVLELSASASATEIRQAYRRLAQLTHPDRL--EHGDGTRFIRLTQAYEILGDEEQRKAYN']
Generated sequences saved to generated_vect_sequences_Jcond_n_PCA50_n_bins35PCA_comp
